# synthetic pipeline condensed-02 03


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_pipeline_condensed-02_03_code_reference.md`


In [ ]:
import os

from datetime import datetime, timedelta

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.premelt_stage_writer import (
    build_observations_premelt_stage,
    validate_observations_premelt_stage,
)

from utils.synthetic.pipeline.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_observations_timestamped_stage,
    validate_observations_timestamped_stage,
)


In [ ]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "synthetic_run_001")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")
ASSET_ID = os.getenv("SYNTHETIC_ASSET_ID", "pump_asset_001")

IF_EXISTS_FLAG = str("replace")
RANDOM_SEED = int(42)
NUMBER_OF_SENSORS = int(52)
CHUNK_SIZE = int(50000)

PREMELT_SOURCE_TABLE_NAME = str("synthetic_pump_stream")
PREMELT_DESTINATION_TABLE_NAME = str("synthetic_observations_premelt_stage")

SIMULATION_TIME_CONFIG_TABLE_NAME = str("simulation_timing_config")
SIMULATION_START_DATETIME = str("2026-04-16 00:00:00+00:00")
SIMULATION_SAMPLING_INTERVAL_SECONDS = float(60.0)

TIMESTAMPED_SOURCE_TABLE_NAME = str("synthetic_observations_premelt_stage")
TIMESTAMPED_DESTINATION_TABLE_NAME = str("synthetic_observations_timestamped_stage")

---

In [3]:
engine = get_engine_from_env()


---

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"Starting Observation Step at {formatted_datetime}")

In [ ]:
# =========================================================
# Emergency fix: add missing dropped sensor columns to source table
# =========================================================
# Premelt expects the full 52-sensor schema.
# These dropped sensors were not modeled/generated, so we restore them as NULLs.

from sqlalchemy import text

# sensor_15 and sensor_50 were dropped during profiling and never generated;
# premelt expects the full 52-column schema, so restore them as NULL columns.
missing_sensor_columns_for_premelt = ["sensor_15", "sensor_50"]

with engine.begin() as conn:
    for sensor_col in missing_sensor_columns_for_premelt:
        conn.execute(
            text(
                f"""
                ALTER TABLE {SCHEMA}.{PREMELT_SOURCE_TABLE_NAME}
                ADD COLUMN IF NOT EXISTS {sensor_col} DOUBLE PRECISION;
                """
            )
        )

print(
    "Added missing premelt source columns if needed:",
    missing_sensor_columns_for_premelt,
)

In [4]:

premelt_table_name = build_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=PREMELT_SOURCE_TABLE_NAME,
    target_table=PREMELT_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
            asset_id=ASSET_ID,
    if_exists=IF_EXISTS_FLAG,
)

print("Built table:", premelt_table_name)


Built table: synthetic_observations_premelt_stage


---

In [5]:

validation_dataframe = validate_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=PREMELT_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,min_observation_index,max_observation_index,distinct_generated_row_id_count,min_batch_id,max_batch_id
0,72000,1,72000,72000,1,1


---

In [6]:
# Inspection

inspection_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        batch_id,
        row_in_batch,
        global_cycle_id,
        stream_state,
        phase,
        meta_episode_id,
        is_telemetry_event,
        telemetry_event_type,
        producer_send_attempt
    FROM capstone.synthetic_observations_premelt_stage
    ORDER BY observation_index
    LIMIT 25
    """
)

display(inspection_dataframe)

,dataset_id,run_id,asset_id,generated_row_id,observation_index,batch_id,row_in_batch,global_cycle_id,stream_state,phase,meta_episode_id,is_telemetry_event,telemetry_event_type,producer_send_attempt
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000001,1,1,0,1,normal,normal,0,False,None,1
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000002,2,1,1,2,normal,normal,0,False,None,1
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000003,3,1,2,3,normal,normal,0,False,None,1
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000004,4,1,3,4,normal,normal,0,False,None,1
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000005,5,1,4,5,normal,normal,0,False,None,1
5,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000006,6,1,5,6,normal,normal,0,False,None,1
6,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000007,7,1,6,7,normal,normal,0,False,None,1
7,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000008,8,1,7,8,normal,normal,0,False,None,1
8,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000009,9,1,8,9,normal,normal,0,False,None,1
9,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000010,10,1,9,10,normal,normal,0,False,None,1


---

In [7]:
# Dispose Engine
engine.dispose()

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"End of Observation Step at {formatted_datetime}")

---

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"Start of Timestamping Step at {formatted_datetime}")

In [8]:
engine = get_engine_from_env()

---

In [9]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


'simulation_timing_config'

---

In [10]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    # Deactivate any prior timing config for this run before inserting a new
    # one; only one timing config row should be active per dataset_id/run_id pair.
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

Timing config ready.


---

In [11]:
timestamped_table_name = build_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


[chunk] 1 | source rows 0 to 9,999
[timestamp] wrote chunk 1 with 10,000 rows
[chunk] 2 | source rows 10,000 to 19,999
[timestamp] wrote chunk 2 with 10,000 rows
[chunk] 3 | source rows 20,000 to 29,999
[timestamp] wrote chunk 3 with 10,000 rows
[chunk] 4 | source rows 30,000 to 39,999
[timestamp] wrote chunk 4 with 10,000 rows
[chunk] 5 | source rows 40,000 to 49,999
[timestamp] wrote chunk 5 with 10,000 rows
[chunk] 6 | source rows 50,000 to 59,999
[timestamp] wrote chunk 6 with 10,000 rows
[chunk] 7 | source rows 60,000 to 69,999
[timestamp] wrote chunk 7 with 10,000 rows
[chunk] 8 | source rows 70,000 to 71,999
[timestamp] wrote chunk 8 with 2,000 rows
Built table: synthetic_observations_timestamped_stage


---

In [12]:
validation_dataframe = validate_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,min_observation_timestamp,max_observation_timestamp,distinct_observation_timestamp_count
0,72000,72000,2026-04-07 00:00:00+00:00,2026-05-26 23:59:00+00:00,72000


---

In [13]:

sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        generated_row_id,
        stream_state,
        phase,
        meta_episode_id,
        sensor_00,
        sensor_01
    FROM capstone.synthetic_observations_timestamped_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sample_dataframe)


,observation_index,observation_timestamp,generated_row_id,stream_state,phase,meta_episode_id,sensor_00,sensor_01
0,1,2026-04-07 00:00:00+00:00,premelt_run_001_obs_000000000001,normal,normal,0,2.323981,49.510297
1,2,2026-04-07 00:01:00+00:00,premelt_run_001_obs_000000000002,normal,normal,0,2.313855,44.928199
2,3,2026-04-07 00:02:00+00:00,premelt_run_001_obs_000000000003,normal,normal,0,2.466675,45.637556
3,4,2026-04-07 00:03:00+00:00,premelt_run_001_obs_000000000004,normal,normal,0,2.469336,46.821980
4,5,2026-04-07 00:04:00+00:00,premelt_run_001_obs_000000000005,normal,normal,0,1.985757,45.531660
5,6,2026-04-07 00:05:00+00:00,premelt_run_001_obs_000000000006,normal,normal,0,1.877601,46.288792
6,7,2026-04-07 00:06:00+00:00,premelt_run_001_obs_000000000007,normal,normal,0,2.457292,45.652845
7,8,2026-04-07 00:07:00+00:00,premelt_run_001_obs_000000000008,normal,normal,0,2.477133,45.519685
8,9,2026-04-07 00:08:00+00:00,premelt_run_001_obs_000000000009,normal,normal,0,2.451138,46.494561
9,10,2026-04-07 00:09:00+00:00,premelt_run_001_obs_000000000010,normal,normal,0,2.198815,50.478982


----

In [14]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_observations_timestamped_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(timestamp_check_dataframe)


,observation_index,observation_timestamp,stream_state,phase,meta_episode_id
0,1,2026-04-07 00:00:00+00:00,normal,normal,0
1,2,2026-04-07 00:01:00+00:00,normal,normal,0
2,3,2026-04-07 00:02:00+00:00,normal,normal,0
3,4,2026-04-07 00:03:00+00:00,normal,normal,0
4,5,2026-04-07 00:04:00+00:00,normal,normal,0
5,6,2026-04-07 00:05:00+00:00,normal,normal,0
6,7,2026-04-07 00:06:00+00:00,normal,normal,0
7,8,2026-04-07 00:07:00+00:00,normal,normal,0
8,9,2026-04-07 00:08:00+00:00,normal,normal,0
9,10,2026-04-07 00:09:00+00:00,normal,normal,0


----

In [15]:
# Dispose Engine
engine.dispose()

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"End of Timestamping Step at {formatted_datetime}")

---

In [ ]:
# Reference: https://stackoverflow.com/questions/49817409/running-a-jupyter-notebook-from-another-notebook

%run ./synthetic_pipeline_testing_export_csv.ipynb

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

This condensed support notebook documents a shortened premelt/timestamp workflow path.
